## 1.提示模板

### 1.1 PromptTemplate 字符串提示模板

#### 1.1.1 基础用法

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

# 1. 实例化提示词模版对象
# 1.1 方式一 通过类的类方法from_template(推荐)
prompt_template = PromptTemplate.from_template(template="你是一个{topic}的专家")
# 1.2 方式二直接实例化
# prompt_template= PromptTemplate(template="你是一个{topic}的专家")


# 2. 格式化提示词(赋值)
# 2.1 invoke()格式化---用的最多
# 2.2 format_prompt()格式化提示词---用的一般多
# 2.3 format()----Python底层用来格式化方法
prompt_value = prompt_template.invoke({"topic": "Python领域"})  # dict

print(prompt_value)  # StringPromptValue(text:) # 期望的：你是一个Python领域的专家



### 金标准：几乎所有的LangChain组件都实现了Runnable接口，所以都可以调用该接口提供的统一标准方法invoke.Runnable接口中会统暴露一系列标准方法。供外部调用

# 几乎：模型实例对象组件/提示词模版对象组件/记忆对象组件/Chains对象组件/工具对象组件/Agent对象组件


text='你是一个Python领域的专家'


#### 1.1.2 部分变量

In [9]:
# 方式一：调用partial方法固定部分变量
# prompt = PromptTemplate.from_template(
#     "讲一个关于{topic}的{adjective}故事"
# )
# fixed_prompt = prompt.partial(adjective="有趣的")
# print(fixed_prompt)
# print(fixed_prompt.invoke({"topic": "编程"}))


# 方式二：创建时直接指定partial_variables
prompt = PromptTemplate(
    template="请解释{concept}，使用{style}风格",
    input_variables=["concept"],
    partial_variables={"style": "简单易懂"}
)
print(prompt.invoke({"concept": "递归"}))

text='请解释递归，使用简单易懂风格'


### 1.2  ChatPromptTemplate 聊天提示模板

#### 1.2.1 基础用法

In [11]:
from langchain_core.prompts import ChatPromptTemplate

# 使用from_messages构造（推荐）
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个专业的{role}。"),
    ("human", "请回答关于{topic}的问题。"),
    ("ai", "好的，我会尽力回答。"),
    ("human", "{question}")
])

# 调用模板，生成消息列表---ChatPromptValue(message属性)----StringPromptValue和ChatPromptValue属于平级关系
messages = chat_prompt.invoke({
    "role": "Python编程助手",
    "topic": "Python装饰器",
    "question": "什么是装饰器？"
})
print(messages)

messages=[SystemMessage(content='你是一个专业的Python编程助手。', additional_kwargs={}, response_metadata={}), HumanMessage(content='请回答关于Python装饰器的问题。', additional_kwargs={}, response_metadata={}), AIMessage(content='好的，我会尽力回答。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='什么是装饰器？', additional_kwargs={}, response_metadata={})]


#### 1.2.2 使用Message对象

In [13]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="你是一个有帮助的助手"),    # 固定内容用Message对象
    HumanMessage(content="你好！"),                     # 固定内容
    AIMessage(content="你好！有什么可以帮助你的？"),      # 固定内容
    ("human", "请介绍{topic}")                          # 含变量的用元组形式
])

messages = chat_prompt.invoke({"topic": "LangChain","what":"AI"})
print(messages)

messages=[SystemMessage(content='你是一个有帮助的{what}助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好！', additional_kwargs={}, response_metadata={}), AIMessage(content='你好！有什么可以帮助你的？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='请介绍LangChain', additional_kwargs={}, response_metadata={})]


#### 1.2.3 MessagesPlaceholder

In [20]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是AI助手"),
    MessagesPlaceholder(variable_name="history"),   # 对话历史插槽
    ("human", "{input}")
])

# 调用时传入历史消息列表
messages = prompt.invoke({
    "history": [
        HumanMessage(content="什么是Python？"),
        AIMessage(content="Python是一种通用编程语言。"),
    ],
    "input": "它有什么特点？"
})
print(messages)

messages=[SystemMessage(content='你是AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='什么是Python？', additional_kwargs={}, response_metadata={}), AIMessage(content='Python是一种通用编程语言。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='它有什么特点？', additional_kwargs={}, response_metadata={})]


In [23]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "你是AI助手"),
    ("placeholder","{history}"), # 简化写法
    ("human", "{input}")
])

# 调用时传入历史消息列表
messages = prompt.invoke({
    "history": [
        HumanMessage(content="什么是Python？"),
        AIMessage(content="Python是一种通用编程语言。"),
    ],
    "input": "它有什么特点？"
})
print(messages)

ValueError: Unexpected message type: abc. Use one of 'human', 'user', 'ai', 'assistant', or 'system'.

### 1.3 FewShotPromptTemplate 少样本提示模

#### 1.3.1 基本用法

In [26]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# 第一步：准备示例数据(结合业务构建对应的示例)
examples = [
    {"input": "高兴", "output": "开心"},
    {"input": "难过", "output": "悲伤"},
    {"input": "生气", "output": "愤怒"}
]
# 第二步：定义单条示例的格式化模板
example_formatter = PromptTemplate(
    template="输入: {input}\n输出: {output}",
    input_variables=["input", "output"]
)
# 第三步：创建少样本提示模板
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_formatter,
    prefix="以下是一些同义词转换的例子：",      # 示例前的说明文字
    suffix="\n输入: {input}\n输出:",         # 示例后的实际问题
    input_variables=["input"]
)

# 调用
print(few_shot_prompt.invoke({"input": "兴奋"}))

# FewShotPromptTemplate.invoke之后返回的PromptValue是 StringPromptValue

text='以下是一些同义词转换的例子：\n\n输入: 高兴\n输出: 开心\n\n输入: 难过\n输出: 悲伤\n\n输入: 生气\n输出: 愤怒\n\n\n输入: 兴奋\n输出:'


#### 1.3.2 对接LLM

In [29]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain.chat_models import init_chat_model
# 第一步：准备示例数据(结合业务构建对应的示例)
examples = [
    {"input": "高兴", "output": "开心"},
    {"input": "难过", "output": "悲伤"},
    {"input": "生气", "output": "愤怒"}
]
# 第二步：定义单条示例的格式化模板
example_formatter = PromptTemplate(
    template="输入: {input}\n输出: {output}",
    input_variables=["input", "output"]
)
# 第三步：创建少样本提示模板
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_formatter,
    prefix="以下是一些同义词转换的例子：",      # 示例前的说明文字
    suffix="\n输入: {input}\n输出:",         # 示例后的实际问题
    input_variables=["input"]
)

# 第四步: 格式化提示词模版对象
prompt_value=few_shot_prompt.invoke({"input": "兴奋"})

# 第五步：实例化LLM模型
llm= init_chat_model(model="gpt-4o",model_provider="openai")

# 第六步：调用
reps=llm.invoke(prompt_value)

print(reps.content)
# 输出：

激动
